# Food reviews Analysis

In this notebook, from the cleaned and then labelled reviews collection, the main insights useful to the analysis are extracted. Going from the numerical and tabular exploration of the reviews, to the sentiment analysis.

Some main questions we want to answer are:
- What are the most reviewed products?
- Which products perform best or worst?
- How do reviews behave over time?
- Who are the users?

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/Reviews_clean.csv")
print(df.shape)

(395159, 13)


### Main insight about products and users

In [9]:
top_products = (
    df.groupby('ProductId')
    .agg({
        'Summary': 'first',
        'Text': 'first',
        'UserId': 'count'
    })
    .rename(columns={'UserId': 'review_count'})
    .sort_values(by='review_count', ascending=False)
)

print(top_products.head(20))

                                                      Summary  \
ProductId                                                       
B007JFMH8M                                         Delicious!   
B002QWP89S    addictive! but works for night coughing in dogs   
B003B3OOPA                      Taste good, great with honey!   
B001EO5Q64       GREAT COCONUT OIL....Try it, you'll like it!   
B0013NUGDE                                      Yummy snacks!   
B000KV61FC                  The best dog toy I ever bought :)   
B005K4Q37A                 Low Quality. High Calorie and Fat.   
B000UBD88A  senseo is the best taste of all the coffees av...   
B000NMJWZO                                        GF Greatest   
B0090X8IPM                 Not bitter and easy on my stomach!   
B005ZBZLT4                      Not as Bold as advertised  ;(   
B006MONQMC     Light and flavorful; helps me drink more water   
B002IEZJMA                                 Pretty good coffee   
B002IEVJRY  Needs to be r

### Sentiment Analysis

Afterwards, a sentiment analysis is performed on the reviews, in order to understand the overall sentiment of the reviews and how it relates to the products and users. The sentiment analysis is performed using a pre-trained BERT model fine-tuned on the food reviews dataset. The model is then evaluated on a test set of 300 reviews, and the results are compared to the true labels.

In [ ]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

test_df = pd.read_csv("../data/processed/test_data.csv").sample(frac=1, random_state=42).head(300)
model = pipeline("sentiment-analysis", model="../src/models/bert-reviews-tuned", tokenizer="../src/models/bert-reviews-tuned", truncation=True)
label_map = {
    'LABEL_0': 0,
    'LABEL_1': 1,
    'LABEL_2': 2
}

results = model(test_df['text'].tolist())
test_df['predicted_sentiment'] = [r['label'] for r in results]

test_df.to_csv("../data/processed/test_results.csv", index=False)

# metrics
y_true = test_df['label'].astype(int).tolist()
y_pred = test_df['predicted_sentiment'].map(label_map).astype(int).tolist()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1}")

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6508.62it/s]


Accuracy: 0.8633333333333333
Precision: 0.6561830933923957
Recall: 0.6206756206756207
F1-Score: 0.628971941362952
Confusion Matrix:
[[ 31   3   8]
 [  7   4  13]
 [  5   5 224]]


In [ ]:
results_df = pd.DataFrame({
    'text': test_df['text'],
    'true_label': y_true,
    'pred_label': y_pred
})
misclassified = results_df[results_df['true_label'] != results_df['pred_label']]

pd.set_option('display.max_colwidth', 700)
print(misclassified.head(10))

Totale errori: 41
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              text  \
16157                                                                                                                                                                                                                                                                           

An analysis of a sample of the misclassified reviews is performed, in order to understand the main reasons of the misclassification. To a fast inspection, it seems that the model misclassifies mainly reviews that are not clearly positive or negative, but rather neutral or mixed.  
  
For example some of them, like id 12374, are reviews that are positive about the product but negative about the delivery, or vice versa.  
Others, like id 31378, clearly express in the text which score they are giving, but mention a lot of other contrasting opinions (such as flavours they love and hate). This kind of reviews seems to have confused the model, which is not able to understand the overall sentiment of the review.

### Sentiment Explanation

In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained('../src/models/bert-reviews-tuned', num_labels=3)
tokenizer = AutoTokenizer.from_pretrained('../src/models/bert-reviews-tuned')

from transformers_interpret import SequenceClassificationExplainer

cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer,
)
word_attributions = cls_explainer("This stuff is awful, cheap leaves and just not good. I've made better tea drying blackberry leaves! Stick to Liptons or Red Rose!!")
print(f'Predicted label {cls_explainer.predicted_class_index} aka {cls_explainer.predicted_class_name} label')
_ = cls_explainer.visualize()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5181.40it/s]


Predicted label 0 aka LABEL_0 label


In [6]:
import shap
import transformers

# Carica il modello e il tokenizer
model = transformers.AutoModelForSequenceClassification.from_pretrained("../src/models/bert-reviews-tuned", num_labels=3)
tokenizer = transformers.AutoTokenizer.from_pretrained("../src/models/bert-reviews-tuned")

# Crea una pipeline di Hugging Face
pred_pipeline = transformers.pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

# Crea l'explainer di SHAP
explainer = shap.Explainer(pred_pipeline)

# Esegui l'analisi su un esempio
text = ["This stuff is awful, cheap leaves and just not good. I've made better tea drying blackberry leaves! Stick to Liptons or Red Rose!!"]
shap_values = explainer(text)

# Visualizza
shap.plots.text(shap_values[0])

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2481.26it/s]
PartitionExplainer explainer: 2it [01:14, 74.20s/it]               
